# 03 — Distributed Gemini judge và final merge (LOCAL)

B/C/D chọn `ACTION='judge'` để mỗi người chấm 200 answers của config mình. A chọn `ACTION='merge'` sau khi nhận ba judge-shard ZIP; merge không gọi Gemini lại.

Một answer thành công dùng **một Gemini API call** và nhận cả ba score trong cùng JSON. `retry_attempts=3` chỉ áp dụng khi call lỗi.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = None
ACTION = 'judge'  # judge | merge
RUNNER = 'D'      # B | C | D khi ACTION='judge'
MINIMUM_KEY_COUNT = 2

RUNNER_CONFIGS = {
    'B': 'report_shortlist_3::graph_dense_rrf',
    'C': 'report_shortlist_3::semantic_gs_rrf_rerank_k40',
    'D': 'report_shortlist_3::semantic_gs_rrf_no_rerank_reference',
}
KEY_OFFSETS = {'B': 0, 'C': 1, 'D': 2}
assert ACTION in {'judge', 'merge'}
assert RUNNER in RUNNER_CONFIGS

def find_repo(explicit=None):
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        assert (candidate / 'backend' / 'app').exists(), candidate
        return candidate
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    direct = [p for p in starts if (p / 'backend' / 'app').exists()]
    children = [p / 'tuvi-battu-graphrag' for p in starts if (p / 'tuvi-battu-graphrag' / 'backend' / 'app').exists()]
    matches = sorted(set(direct + children))
    assert len(matches) == 1, f'Hãy đặt REPO_ROOT; tìm thấy {matches}'
    return matches[0]

REPO_ROOT = find_repo(REPO_ROOT)
KIT_ROOT = REPO_ROOT / 'benchmark' / 'tuvi_golden_dataset' / 'local_llm_ablation'
BUNDLE_DIR = KIT_ROOT / 'artifacts' / 'context_bundle_v1'
assert (BUNDLE_DIR / 'bundle_manifest.json').exists(), BUNDLE_DIR
sys.path.insert(0, str(KIT_ROOT))


In [2]:
if ACTION == 'judge':
    prediction_dir = KIT_ROOT / 'artifacts' / 'judge_inputs' / RUNNER
    output_dir = KIT_ROOT / 'artifacts' / 'gemini_judge_shards' / RUNNER
    judge_config = {
        'repo_root': str(REPO_ROOT),
        'bundle_dir': str(BUNDLE_DIR),
        'prediction_roots': [str(prediction_dir)],
        'suites': ['report_shortlist_3'],
        'selected_config_keys': [RUNNER_CONFIGS[RUNNER]],
        'expected_model_ids': ['Qwen/Qwen2.5-7B-Instruct', 'google/gemma-3-4b-it'],
        'judge_model': 'gemini-3.1-flash-lite-preview',
        'initial_key_offset': KEY_OFFSETS[RUNNER],
        'minimum_key_count': MINIMUM_KEY_COUNT,
        'retry_attempts': 3,
        'retry_base_seconds': 2.0,
        'retry_failed': True,
        'allow_incomplete': False,
        'shard_name': RUNNER,
        'output_dir': str(output_dir),
    }
    from local_tools.run_judge import run_gemini_judge
    result = run_gemini_judge(judge_config)
else:
    shard_dir = KIT_ROOT / 'artifacts' / 'downloaded_judge_shards'
    output_dir = KIT_ROOT / 'artifacts' / 'gemini_judge_final'
    merge_config = {
        'repo_root': str(REPO_ROOT),
        'kit_root': str(KIT_ROOT),
        'bundle_dir': str(BUNDLE_DIR),
        'judge_shard_roots': [str(shard_dir)],
        'suites': ['report_shortlist_3'],
        'expected_model_ids': ['Qwen/Qwen2.5-7B-Instruct', 'google/gemma-3-4b-it'],
        'output_dir': str(output_dir),
    }
    from local_tools.merge_judge_shards import merge_gemini_judge_shards
    result = merge_gemini_judge_shards(merge_config)
result


C:\Users\Khanh Linh\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\UNI_STUDY\Year3\Semester3\TextMining\tuvi-battu-graphrag\backend\app\rag\evaluation.py:221: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


judged=10/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=20/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=30/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=40/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=50/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=60/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=70/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=80/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=90/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=100/200 model=Qwen/Qwen2.5-7B-Instruct status=completed
judged=110/200 model=google/gemma-3-4b-it status=completed
judged=120/200 model=google/gemma-3-4b-it status=completed
judged=130/200 model=google/gemma-3-4b-it status=completed
judged=140/200 model=google/gemma-3-4b-it status=completed
judged=150/200 model=google/gemma-3-4b-it status=completed
judged=160/200 model=google/gemma-3-4b-it status=completed
judged=170/200 model=goog

{'schema_version': 'local-llm-gemini-judge-v2',
 'started_at': '2026-08-31T03:49:24.647756+00:00',
 'completed_at': '2026-08-31T04:07:39.879451+00:00',
 'judge_backend': 'gemini',
 'judge_model': 'gemini-3.1-flash-lite-preview',
 'shard_name': 'd',
 'selected_suites': ['report_shortlist_3'],
 'selected_config_keys': ['report_shortlist_3::semantic_gs_rrf_no_rerank_reference'],
 'expected_model_ids': ['Qwen/Qwen2.5-7B-Instruct', 'google/gemma-3-4b-it'],
 'retrieval_pair_count': 100,
 'expected_prediction_count': 200,
 'completed_prediction_count': 200,
 'missing_prediction_count': 0,
 'unexpected_prediction_count': 0,
 'judged_completed_count': 200,
 'judged_failed_count': 0,
 'is_complete': True,
 'prediction_files': ['D:\\UNI_STUDY\\Year3\\Semester3\\TextMining\\tuvi-battu-graphrag\\benchmark\\tuvi_golden_dataset\\local_llm_ablation\\artifacts\\judge_inputs\\D\\gemma3_4b\\predictions_shard_gemma3_4b.jsonl',
  'D:\\UNI_STUDY\\Year3\\Semester3\\TextMining\\tuvi-battu-graphrag\\benchmark\

In [3]:
if ACTION == 'judge':
    assert result['retrieval_pair_count'] == 100, result
    assert result['expected_prediction_count'] == 200, result
    assert result['judged_completed_count'] == 200, result
    assert result['judged_failed_count'] == 0, result
    assert result['is_complete'], result
    assert result['key_rotation']['key_count'] >= MINIMUM_KEY_COUNT, result
    print('PASS — gửi judge-shard ZIP này cho A:', result['archive_path'])
else:
    assert result['source_shard_count'] == 3, result
    assert result['expected_pair_count'] == 600, result
    assert result['completed_pair_count'] == 600, result
    assert result['failed_pair_count'] == 0, result
    assert result['config_result_count'] == 6, result
    assert result['is_complete'], result
    print('PASS — canonical final report:', output_dir / 'evaluation_report.json')


PASS — gửi judge-shard ZIP này cho A: D:\UNI_STUDY\Year3\Semester3\TextMining\tuvi-battu-graphrag\benchmark\tuvi_golden_dataset\local_llm_ablation\artifacts\gemini_judge_shards\gemini_judge_shard_d.zip
